# 9장. 회귀 분석으로 숫자 예측하기

이 노트북은 `book/chapters/ch09_regression_analysis.md`와 `src/regression.py`의 회귀 분석 흐름을 그대로 실행합니다.

핵심은 높은 성능을 만드는 것이 아니라 다음 원칙을 지키는 것입니다.

- 예측 시점을 먼저 정의합니다.
- 주문 상세의 수량·단가·금액은 목표값 생성에만 사용합니다.
- 식별자와 예측 이후 정보를 입력값에서 제외합니다.
- 날짜 순서로 훈련·테스트 데이터를 나눕니다.
- 단순 평균 베이스라인보다 실제로 나은지 확인합니다.
- 낮거나 음수인 R²도 숨기지 않고 해석합니다.


## 0. 실행 전 확인

이 노트북은 5장에서 만든 다음 파일을 사용합니다.

- `data/processed/customers_clean.csv`
- `data/processed/orders_clean.csv`
- `data/processed/order_items_clean.csv`

파일이 없다면 프로젝트 루트에서 다음 명령을 먼저 실행합니다.

```bash
python scripts/preprocess_data.py
```


## 1. 회귀 문제와 예측 시점

예측 대상은 주문별 주문 상세 금액 합계인 `order_total`입니다.

교육용 예측 시점에서는 주문 시점과 결제 수단, 고객의 비식별 특성은 알 수 있지만 주문 상세의 수량·단가·금액은 모델에 제공하지 않는다고 가정합니다.

따라서 `quantity`, `unit_price`, `line_total`, `item_count`, `total_quantity`, `avg_unit_price`, `order_status`, `order_id`, `customer_id`는 입력값으로 사용하지 않습니다.


## 2. 패키지와 프로젝트 경로 설정


In [31]:
from pathlib import Path
import sys

import pandas as pd

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"

from src.regression import (
    CATEGORICAL_FEATURES,
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    NUMERIC_FEATURES,
    TARGET_COLUMN,
    build_leakage_checklist,
    build_regression_dataset,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print("프로젝트 루트:", PROJECT_ROOT.resolve())
print("전처리 데이터:", PROCESSED_DIR.resolve())
print("보고서 폴더:", REPORT_DIR.resolve())


프로젝트 루트: C:\dev\llm-data-analysis-course
전처리 데이터: C:\dev\llm-data-analysis-course\data\processed
보고서 폴더: C:\dev\llm-data-analysis-course\reports


## 3. 전처리 데이터 불러오기

원본 CSV로 자동 대체하지 않습니다. 5장의 전처리 기준과 다른 데이터가 섞이면 실습 결과를 비교하기 어렵기 때문입니다.


In [32]:
data = load_regression_source_data(PROCESSED_DIR)

customers = data["customers"]
orders = data["orders"]
order_items = data["order_items"]

print("customers:", customers.shape)
print("orders:", orders.shape)
print("order_items:", order_items.shape)


customers: (150, 6)
orders: (300, 7)
order_items: (764, 6)


## 4. 누수 없는 주문 단위 모델링 데이터 만들기

`order_total`은 주문 상세에서 계산하지만, 주문 상세에서 만든 다른 집계값은 모델 입력에 포함하지 않습니다.


In [33]:
model_data = build_regression_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

print("모델링 데이터:", model_data.shape)
display(
    model_data[
        [
            "order_id",
            "order_date",
            *FEATURE_COLUMNS,
            TARGET_COLUMN,
        ]
    ].head()
)


모델링 데이터: (300, 11)


,order_id,order_date,order_month,order_dayofweek,age,payment_method,gender,city,order_total
0,238,2025-09-14,9,6,67,bank_transfer,M,대구,361000
1,79,2025-09-16,9,1,63,bank_transfer,F,광주,283000
2,12,2025-09-17,9,2,61,kakao_pay,F,인천,444000
3,169,2025-09-17,9,2,69,kakao_pay,M,서울,448000
4,152,2025-09-18,9,3,63,naver_pay,F,광주,985000


In [34]:
print("숫자형 입력값:", NUMERIC_FEATURES)
print("범주형 입력값:", CATEGORICAL_FEATURES)
print("예측 대상:", TARGET_COLUMN)
print("금지 입력값:", sorted(FORBIDDEN_FEATURES))

validate_feature_columns(FEATURE_COLUMNS)


숫자형 입력값: ['order_month', 'order_dayofweek', 'age']
범주형 입력값: ['payment_method', 'gender', 'city']
예측 대상: order_total
금지 입력값: ['avg_unit_price', 'customer_id', 'item_count', 'line_total', 'order_id', 'order_status', 'order_total', 'quantity', 'total_quantity', 'unit_price']


## 5. 날짜 순서로 훈련·테스트 데이터 분할

과거 주문으로 이후 주문을 예측하는 상황을 모방하기 위해 무작위 분할 대신 날짜 순서 분할을 사용합니다.


In [35]:
train_data, test_data = split_model_data_by_time(
    model_data,
    test_size=0.2,
)

X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()

print(
    "훈련 기간:",
    train_data["order_date"].min(),
    "~",
    train_data["order_date"].max(),
)
print(
    "테스트 기간:",
    test_data["order_date"].min(),
    "~",
    test_data["order_date"].max(),
)
print("훈련:", X_train.shape, "테스트:", X_test.shape)


훈련 기간: 2025-09-14 00:00:00 ~ 2026-06-14 00:00:00
테스트 기간: 2026-06-14 00:00:00 ~ 2026-09-13 00:00:00
훈련: (240, 6) 테스트: (60, 6)


## 6. 베이스라인·선형 회귀·랜덤 포레스트 비교

각 모델의 전처리 파이프라인은 훈련 데이터 안에서만 결측치 대체, 표준화, 원-핫 인코딩을 학습합니다.

`Baseline Mean`은 훈련 목표값의 평균을 모든 테스트 주문에 예측합니다. 다른 모델은 최소한 이 기준과 비교해야 합니다.


In [36]:
models, model_comparison, predictions = (
    train_and_evaluate_models(
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        random_state=42,
    )
)

model_comparison


,model,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,412811.979167,455895.833333,531015.673566,-0.002361,0.00
1,Linear Regression,396341.863565,484513.141613,568269.248676,-0.147936,-6.28
2,Random Forest,323900.886337,484884.127521,564053.232728,-0.130966,-6.36


`MAE_improvement_vs_baseline_pct`가 양수이면 베이스라인보다 평균 절대 오차가 줄었습니다. 음수이면 복잡한 모델이 단순 평균보다도 나쁘다는 뜻입니다.

훈련 MAE가 매우 낮고 테스트 MAE가 크면 과적합 가능성을 확인합니다.


## 7. 시간 순서 교차검증

한 번의 테스트 기간 결과만 믿지 않고, 훈련 기간 안에서 여러 시점으로 나누어 성능의 평균과 변동을 확인합니다.


In [37]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
    max_splits=5,
)

cv_summary


,model,cv_MAE_mean,cv_MAE_std,cv_R2_mean
0,Random Forest,430352.937414,3649.627596,-0.149754
1,Linear Regression,524293.976203,113293.799328,-0.724768


교차검증 MAE 표준편차가 크면 기간에 따라 성능이 불안정할 수 있습니다. R²가 반복적으로 음수라면 현재 입력값으로 평균 예측을 안정적으로 넘지 못할 가능성이 큽니다.


## 8. 실제값·예측값과 잔차 확인

테스트 MAE가 가장 낮은 비베이스라인 모델을 진단용으로 선택합니다. 선택된 모델이 베이스라인보다 낫다는 뜻은 아니므로 비교표를 함께 확인해야 합니다.


In [38]:
selected_model_name = select_diagnostic_model(
    model_comparison
)

prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)

print("진단 모델:", selected_model_name)
prediction_result.head(10)


진단 모델: Linear Regression


,order_id,order_date,actual_order_total,predicted_order_total,residual,abs_error,model
0,51,2026-08-21,1877000,6.140507e+05,1.262949e+06,1.262949e+06,Linear Regression
1,168,2026-08-17,10000,9.941418e+05,-9.841418e+05,9.841418e+05,Linear Regression
2,73,2026-08-20,72000,1.006061e+06,-9.340606e+05,9.340606e+05,Linear Regression
3,183,2026-07-29,1685000,7.667861e+05,9.182139e+05,9.182139e+05,Linear Regression
4,15,2026-07-29,1677000,7.837536e+05,8.932464e+05,8.932464e+05,Linear Regression
5,145,2026-08-02,1863000,9.751509e+05,8.878491e+05,8.878491e+05,Linear Regression
6,96,2026-07-17,1690000,8.032754e+05,8.867246e+05,8.867246e+05,Linear Regression
7,21,2026-08-25,1767000,8.807951e+05,8.862049e+05,8.862049e+05,Linear Regression
8,179,2026-09-13,86000,9.575597e+05,-8.715597e+05,8.715597e+05,Linear Regression
9,282,2026-06-30,1566000,7.019192e+05,8.640808e+05,8.640808e+05,Linear Regression


In [39]:
prediction_result["abs_error"].describe()


count    6.000000e+01
mean     4.845131e+05
std      2.994520e+05
min      4.731499e+04
25%      2.322634e+05
50%      4.957378e+05
75%      7.196309e+05
max      1.262949e+06
Name: abs_error, dtype: float64

`residual = 실제값 - 예측값`입니다.

- 잔차가 양수이면 실제값을 낮게 예측했습니다.
- 잔차가 음수이면 실제값을 높게 예측했습니다.
- 큰 절대 오차가 특정 기간이나 금액대에 몰리는지 확인합니다.


## 9. 체크리스트와 결과 저장

예측 결과에는 주문 식별자가 포함되므로 `_internal` 파일은 외부 공개용으로 사용하지 않습니다.


In [40]:
regression_checklist = build_leakage_checklist()
regression_checklist


,check_item,status
0,예측 시점이 명확한가?,□
1,목표값과 목표값의 계산 재료를 입력에서 제외했는가?,□
2,예측 이후에 알 수 있는 정보를 사용하지 않았는가?,□
3,식별자를 일반 숫자 변수로 사용하지 않았는가?,□
4,전처리기가 훈련 데이터 안에서만 학습되는가?,□
5,시간 순서 또는 업무 목적에 맞는 분할을 사용했는가?,□
6,단순 베이스라인과 비교했는가?,□
7,테스트 데이터로 최종 성능을 평가했는가?,□
8,"MAE, RMSE, R²를 올바르게 해석했는가?",□
9,음수 R²와 낮은 성능을 숨기지 않았는가?,□


In [41]:
output_paths = save_regression_outputs(
    model_data=model_data,
    train_data=train_data,
    test_data=test_data,
    model_comparison=model_comparison,
    cv_summary=cv_summary,
    prediction_result=prediction_result,
    checklist=regression_checklist,
    selected_model_name=selected_model_name,
    report_dir=REPORT_DIR,
)

for name, path in output_paths.items():
    print(f"{name}: {path}")


model_data_internal: c:\dev\llm-data-analysis-course\reports\ch09_regression_model_data_internal.csv
model_comparison: c:\dev\llm-data-analysis-course\reports\ch09_regression_model_comparison.csv
cv_summary: c:\dev\llm-data-analysis-course\reports\ch09_regression_cv_summary.csv
predictions_internal: c:\dev\llm-data-analysis-course\reports\ch09_regression_predictions_internal.csv
checklist: c:\dev\llm-data-analysis-course\reports\ch09_regression_checklist.csv
report: c:\dev\llm-data-analysis-course\reports\ch09_regression_report.md
actual_vs_predicted_figure: c:\dev\llm-data-analysis-course\reports\figures\ch09_actual_vs_predicted.png
residual_figure: c:\dev\llm-data-analysis-course\reports\figures\ch09_residual_histogram.png


저장되는 주요 결과는 다음과 같습니다.

- 모델 비교표
- 시간 순서 교차검증 요약
- 내부용 모델링 데이터와 예측 오차
- 회귀 분석 체크리스트와 Markdown 보고서
- 실제값·예측값 산점도
- 잔차 히스토그램


## 10. 공통 모듈로 전체 파이프라인 다시 실행하기

앞의 단계는 `src/regression.py`의 `run_regression_analysis()`로 한 번에 재현할 수 있습니다.


In [42]:
regression_result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)

regression_result["model_comparison"]


,model,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,412811.979167,455895.833333,531015.673566,-0.002361,0.00
1,Linear Regression,396341.863565,484513.141613,568269.248676,-0.147936,-6.28
2,Random Forest,323900.886337,484884.127521,564053.232728,-0.130966,-6.36


## 11. 실행 스크립트 사용

프로젝트 루트에서 다음 명령을 실행하면 같은 파이프라인이 수행됩니다.

```bash
python scripts/run_regression_analysis.py
```


## 12. LLM에게 회귀 코드를 요청하는 프롬프트

```text
온라인 쇼핑몰 주문 금액을 예측하는 교육용 회귀 모델을 만들려고 합니다.

예측 대상:
- order_total: 주문별 line_total 합계

예측 시점:
- 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만
  주문 상세의 수량, 단가, 금액은 모델에 제공하지 않음

사용 가능한 입력값:
- payment_method
- order_month
- order_dayofweek
- gender
- age
- city

사용하면 안 되는 입력값:
- order_total, line_total
- quantity, unit_price
- item_count, total_quantity, avg_unit_price
- order_status
- order_id, customer_id

요청:
1. 날짜 순서로 훈련 데이터 80%, 테스트 데이터 20%를 나누어 주세요.
2. 결측치 처리와 OneHotEncoder를 Pipeline 안에 넣어 주세요.
3. DummyRegressor, LinearRegression, RandomForestRegressor를 비교해 주세요.
4. MAE, RMSE, R²와 베이스라인 대비 개선율을 계산해 주세요.
5. 훈련·테스트 성능 차이와 데이터 누수 가능성을 설명해 주세요.
6. R²가 음수일 때의 의미도 설명해 주세요.

주의:
- 실제 데이터에 없는 컬럼을 만들지 마세요.
- 테스트 데이터로 전처리 규칙을 학습하지 마세요.
- 높은 성능을 가정하거나 결과를 임의로 만들어내지 마세요.
```


## 13. 실습 과제

1. `payment_method`를 제외했을 때 베이스라인 대비 성능이 어떻게 달라지는지 비교합니다.
2. `RandomForestRegressor`의 `min_samples_leaf` 값을 바꾸어 훈련·테스트 MAE를 비교합니다.
3. 시간 순서 교차검증에서 성능이 가장 낮은 기간의 데이터 특성을 확인합니다.
4. 예측 시점 이전의 고객 구매 이력을 설계하되 미래 정보가 섞이지 않도록 집계 기준일을 명시합니다.
5. LLM이 제안한 회귀 코드에서 목표값의 계산 재료나 식별자가 입력에 포함됐는지 검토합니다.


In [43]:
# 과제 1. FEATURE_COLUMNS에서 payment_method를 제외한 별도 실험을 설계해 보세요.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

baseline_mae = model_comparison.loc[model_comparison['model'] == 'Baseline Mean', 'test_MAE'].iloc[0]

for label, features in [('전체 입력값', FEATURE_COLUMNS),
                        ('payment_method 제외', [c for c in FEATURE_COLUMNS if c != 'payment_method'])]:
    categorical = [c for c in CATEGORICAL_FEATURES if c in features]
    for name, model in [('Linear Regression', LinearRegression()),
                        ('Random Forest', RandomForestRegressor(random_state=42))]:
        pipe = make_pipeline(
            ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), categorical)], remainder='passthrough'),
            model,
        ).fit(X_train[features], y_train)
        mae = mean_absolute_error(y_test, pipe.predict(X_test[features]))
        print(f'{label} | {name}: test MAE {mae:,.0f} / baseline 대비 {(baseline_mae - mae) / baseline_mae * 100:.2f}%')

전체 입력값 | Linear Regression: test MAE 484,513 / baseline 대비 -6.28%
전체 입력값 | Random Forest: test MAE 506,403 / baseline 대비 -11.08%
payment_method 제외 | Linear Regression: test MAE 470,318 / baseline 대비 -3.16%
payment_method 제외 | Random Forest: test MAE 480,908 / baseline 대비 -5.49%


In [44]:
# 과제 2. RandomForestRegressor의 min_samples_leaf 값을 바꾸어 비교해 보세요.
for leaf in [1, 5, 10, 20, 40]:
    pipe = make_pipeline(
        ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)], remainder='passthrough'),
        RandomForestRegressor(min_samples_leaf=leaf, random_state=42),
    ).fit(X_train, y_train)
    train_mae = mean_absolute_error(y_train, pipe.predict(X_train))
    test_mae = mean_absolute_error(y_test, pipe.predict(X_test))
    print(f'min_samples_leaf={leaf}: train MAE {train_mae:,.0f} / test MAE {test_mae:,.0f}')

min_samples_leaf=1: train MAE 160,559 / test MAE 506,403
min_samples_leaf=5: train MAE 325,249 / test MAE 489,896
min_samples_leaf=10: train MAE 368,440 / test MAE 480,239
min_samples_leaf=20: train MAE 394,983 / test MAE 465,977
min_samples_leaf=40: train MAE 406,752 / test MAE 461,846


In [45]:
# 과제 3. 시간 순서 교차검증에서 성능이 가장 낮은 기간의 데이터 특성을 확인합니다.
from sklearn.model_selection import TimeSeriesSplit

assert train_data['order_date'].is_monotonic_increasing  # 날짜 순서로 정렬되어 있어야 시간 순서 분할이 맞음

rows = []
for fold, (tr, va) in enumerate(TimeSeriesSplit(n_splits=5).split(X_train), start=1):
    pipe = make_pipeline(
        ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES)], remainder='passthrough'),
        LinearRegression(),
    ).fit(X_train.iloc[tr], y_train.iloc[tr])
    va_dates = train_data['order_date'].iloc[va]
    rows.append({
        'fold': fold,
        'valid_period': f'{va_dates.min():%Y-%m-%d} ~ {va_dates.max():%Y-%m-%d}',
        'model_MAE': round(mean_absolute_error(y_train.iloc[va], pipe.predict(X_train.iloc[va]))),
        'baseline_MAE': round(mean_absolute_error(y_train.iloc[va], [y_train.iloc[tr].mean()] * len(va))),
        'train_mean': round(y_train.iloc[tr].mean()),
        'valid_mean': round(y_train.iloc[va].mean()),
        'valid_max': y_train.iloc[va].max(),
    })
display(pd.DataFrame(rows))

,fold,valid_period,model_MAE,baseline_MAE,train_mean,valid_mean,valid_max
0,1,2025-11-09 ~ 2025-12-24,655262,439641,837175,921150,2118000
1,2,2025-12-24 ~ 2026-01-28,670327,382908,879162,947300,2293000
2,3,2026-01-31 ~ 2026-03-17,423754,363062,901875,786025,1981000
3,4,2026-03-19 ~ 2026-04-30,439362,463212,872912,811375,2135000
4,5,2026-05-01 ~ 2026-06-14,432765,423470,860605,840100,1931000


### 과제 3 결과
성능이 가장 낮은 기간은 fold 2(2025-12-24 ~ 2026-01-28)로, model MAE가 670,327원이며 같은 기간 baseline(382,908원)보다 약 75% 크다.

데이터 특성:
- order_month를 숫자로 쓰는데, 검증 기간의 월(fold 1은 12월, fold 2는 1월)이 학습 구간의 월 범위(9~11월, 9~12월) 밖에 있다. 선형 모델의 학습 구간에 1월이 들어간 fold 3부터는 오차가 423,754~439,362원으로 줄었다.
- 학습 구간이 가장 짧아 학습 주문 수가 가장 적다.
- 검증 기간의 평균 주문 금액(921,150원, 947,300원)이 학습 구간 평균(837,175원, 879,162원)보다 높고, fold 2의 최대 주문 금액(2,293,000원)이 가장 크다.

5개 fold 중 모델이 baseline보다 나았던 것은 fold 4 하나뿐이다.

### 과제 4. 예측 시점 이전의 고객 구매 이력 설계
집계 기준일: 각 주문의 주문일 전날까지(주문일 당일 주문은 포함하지 않음).같은 날 주문은 순서를 알 수 없어서, 당일 주문을 포함하면 같은 날의 다른 주문 금액이 섞일 수 있다.

| feature | 정의 | 미래 정보 차단 방법 |
| --- | --- | --- |
| prev_order_count | 이 주문 이전에 같은 고객이 한 주문 수 | 주문일이 현재 주문일보다 앞선 주문만 센다 |
| prev_avg_order_total | 이전 주문들의 평균 주문 금액 | 같은 조건. 첫 주문 고객은 값이 없으므로 훈련 데이터 평균으로 채우고 is_first_order 표시를 함께 둔다 |
| days_since_last_order | 직전 주문 이후 지난 일수 | 같은 조건. 첫 주문은 값이 없음을 별도 표시 |

지켜야 할 원칙:
- 고객의 전체 기간 평균 주문 금액처럼 이후 주문까지 포함한 값은 미래 주문의 금액이 섞이기 때문에 쓰지 않는다. 
- final test 기간의 주문도 그 주문일 이전 주문만으로만 계산한다. 이전 주문이 train 기간에 있어도 주문 시점에 이미 알 수 있는 정보이므로 사용할 수 있다.
- 이전 주문의 상태(취소/환불)는 나중에 바뀔 수 있는데, 데이터에 상태가 바뀐 날짜가 없기 때문에 이력 집계에는 상태와 관계없이 모든 이전 주문을 포함한다.

### 과제 5. LLM이 제안한 회귀 코드 검토
사용한 LLM: ChatGPT / 프롬프트: 노트북 12번 프롬프트

```python

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


# --------------------------------
# 1. 주문별 예측 데이터 만들기
# --------------------------------

# 주문별 line_total 합계로 정답값 생성
order_totals = (
    order_items
    .groupby("order_id", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "order_total"})
)

# 주문 메타데이터 결합
model_data = orders.merge(
    order_totals,
    on="order_id",
    how="inner",
    validate="one_to_one"
)

# 고객의 비식별 특성 결합
model_data = model_data.merge(
    customers[["customer_id", "gender", "age", "city"]],
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# 날짜 형식 확인 및 변환
model_data["order_date"] = pd.to_datetime(
    model_data["order_date"],
    errors="coerce"
)

# 날짜나 정답값이 없는 행 제외
model_data = model_data.dropna(
    subset=["order_date", "order_total"]
).copy()

# 정답값이 숫자인지 확인
model_data["order_total"] = pd.to_numeric(
    model_data["order_total"],
    errors="coerce"
)

model_data = model_data.dropna(
    subset=["order_total"]
).copy()


# --------------------------------
# 2. 입력값과 정답값 지정
# --------------------------------

features = [
    "payment_method",
    "order_month",
    "order_dayofweek",
    "gender",
    "age",
    "city"
]

target = "order_total"

X = model_data[features].copy()
y = model_data[target].copy()


# --------------------------------
# 3. 날짜 순서로 80:20 분리
# --------------------------------

# 날짜별 주문을 한쪽 데이터에만 포함하도록
# 날짜를 기준으로 시간 순서 분리
unique_dates = np.sort(
    model_data["order_date"].unique()
)

split_position = int(len(unique_dates) * 0.8)

cutoff_date = unique_dates[split_position]

train_mask = model_data["order_date"] < cutoff_date
test_mask = model_data["order_date"] >= cutoff_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("훈련 데이터 수:", len(X_train))
print("테스트 데이터 수:", len(X_test))
print("훈련 종료일:", model_data.loc[train_mask, "order_date"].max())
print("테스트 시작일:", model_data.loc[test_mask, "order_date"].min())


# --------------------------------
# 4. 전처리 Pipeline 구성
# --------------------------------

numeric_features = ["age"]

categorical_features = [
    "payment_method",
    "order_month",
    "order_dayofweek",
    "gender",
    "city"
]

# 수치형 데이터: 중앙값으로 결측치 처리
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# 범주형 데이터: 최빈값으로 결측치 처리
# OneHotEncoder로 범주형 데이터 변환
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


# --------------------------------
# 5. 비교할 모델 생성
# --------------------------------

models = {
    "DummyRegressor": DummyRegressor(strategy="mean"),

    "LinearRegression": LinearRegression(),

    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    )
}


# --------------------------------
# 6. 모델 학습 및 평가
# --------------------------------

results = []

for name, model in models.items():

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # 훈련 데이터만으로 전처리 규칙 학습 및 모델 학습
    pipeline.fit(X_train, y_train)

    # 훈련 데이터 예측
    train_pred = pipeline.predict(X_train)

    # 테스트 데이터 예측
    test_pred = pipeline.predict(X_test)

    # 훈련 성능
    train_mae = mean_absolute_error(y_train, train_pred)
    train_rmse = np.sqrt(
        mean_squared_error(y_train, train_pred)
    )
    train_r2 = r2_score(y_train, train_pred)

    # 테스트 성능
    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(
        mean_squared_error(y_test, test_pred)
    )
    test_r2 = r2_score(y_test, test_pred)

    results.append({
        "model": name,
        "train_MAE": train_mae,
        "test_MAE": test_mae,
        "train_RMSE": train_rmse,
        "test_RMSE": test_rmse,
        "train_R2": train_r2,
        "test_R2": test_r2
    })


# --------------------------------
# 7. 성능 비교표 생성
# --------------------------------

results_df = pd.DataFrame(results)

# DummyRegressor의 테스트 MAE를 기준으로 개선율 계산
baseline_mae = results_df.loc[
    results_df["model"] == "DummyRegressor",
    "test_MAE"
].iloc[0]

results_df["baseline_improvement(%)"] = (
    (baseline_mae - results_df["test_MAE"])
    / baseline_mae * 100
)

print("\n모델 성능 비교")
print(results_df.round(3))
```


| 검토 항목 | 확인 방법 | 결과 |
| --- | --- | --- |
| 입력값이 6개뿐인가? | features 목록 확인 | 통과 |
| 목표값 계산 재료가 X에 남아 있지 않은가? | order_total을 만든 뒤 X를 고르는 줄 확인 | 통과. order_items는 정답값 계산에만 사용 |
| 식별자를 숫자 입력값으로 쓰지 않았는가? | X에 들어간 컬럼 확인 | 통과. order_id, customer_id는 병합 키로만 사용 |
| 예측 이후 정보(order_status)를 쓰지 않았는가? | X에 들어간 컬럼 확인 | 통과 |
| 날짜 순서로 나눴는가? | 분할 코드 확인 | 통과. 날짜 기준 컷오프로 나눠 같은 날 주문이 섞이지 않음. train_test_split은 import만 하고 사용하지 않음 |
| 전처리를 Pipeline 안에서 train으로만 학습하는가? | fit 호출 확인 | 통과 |
| 실제 데이터에 없는 컬럼을 만들지 않았는가? | 컬럼 이름과 값 비교 | 확인 필요. orders 파일의 order_month가 연-월 문자열("2025-09")이라면, 노트북의 월 숫자와 의미가 다르고 test의 연-월이 train에 없어 원-핫 인코딩에서 무시된다 |

판단: 수정 후 사용. 누수는 없고 날짜는 기존 제공된 노트북보다 엄격하게 나눠짐. 다만 order_month를 주문일에서 월 숫자로 다시 계산하고, 사용하지 않는 import를 지우고, 고객 병합 후 미매칭 건수를 확인해야 한다.

## 14. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 예측 시점 정의
- 목표값과 목표값 계산 재료 분리
- 식별자와 예측 이후 정보 제외
- 날짜 순서 훈련·테스트 분할
- Pipeline 안에서 결측치 처리와 인코딩
- 평균 베이스라인, 선형 회귀, 랜덤 포레스트 비교
- MAE, RMSE, R²와 베이스라인 대비 개선율 해석
- 시간 순서 교차검증
- 실제값·예측값과 잔차 진단
- 내부 식별 정보 보호
- `src/regression.py`와 실행 스크립트를 통한 재현

낮은 성능도 현재 데이터와 입력 변수의 한계를 알려 주는 유효한 분석 결과입니다.


# Assignment markdown

# Chapter 09 답안 양식. 회귀 분석으로 숫자 예측하기

> 이 내용을 `chapter09.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 배성윤
- GitHub ID: aeuyui
- 작성일: 2026-11-06
- 최종 Notebook URL: https://github.com/aeuyui/llm-data-analysis-study/blob/main/chapter09/chapter09.ipynb

## 1. 예측 문제 정의
- 타깃: order_total (주문별 주문 상세 금액 line_total의 합계)
- 예측 단위: 주문 1건
- 예측 시점: 주문이 생성된 시점. 주문일과 결제 수단, 고객의 비식별 특성은 알지만, 주문에 담긴 상품의 수량·단가·금액은 아직 모른다고 가정.
- 예측 시점에 알 수 있는 정보: payment_method, order_month, order_dayofweek, gender, age, city
- 예측 시점에 알 수 없는 정보:
  - 타깃 계산 재료: quantity, unit_price, line_total, item_count, total_quantity, avg_unit_price
  - 예측 이후에 정해지는 정보: order_status
  - 식별자: order_id, customer_id

### 나의 해석과 판단
order_total은 주문 상세의 line_total을 더한 값이므로, 수량과 단가를 알면 예측이 아니라 정확한 값을 계산하게 된다. 따라서 주문 상세를 알기 전 시점을 예측 시점으로 정해야 모델 예측이 유의미하다. 이 시점에 쓸 수 있는 정보는 결제 수단, 주문 시기, 고객 특성뿐이라 높은 성능은 나오지 않을 것으로 예상된다.

## 2. Leakage 점검
| feature | 예측 시점 사용 가능 | 타깃 계산 재료 | 최종 사용 | 판단 이유 |
| --- | --- | --- | --- | --- |
| payment_method | O | X | 사용 | 주문할 때 고르는 결제 수단 |
| order_month | O | X | 사용 | 주문일에서 계산한 월(숫자 1~12) |
| order_dayofweek | O | X | 사용 | 주문일에서 계산한 요일(숫자 0~6) |
| gender | O | X | 사용 | 주문 전에 알고 있는 고객 속성 |
| age | O | X | 사용 | 주문 전에 알고 있는 고객 속성 |
| city | O | X | 사용 | 주문 전에 알고 있는 고객 속성 |
| quantity, unit_price, line_total | X | O | 제외 | order_total을 직접 계산하는 주문 상세 값 |
| item_count, total_quantity, avg_unit_price | X | O | 제외 | 주문 상세에서 만든 집계값이라 타깃과 같은 재료를 사용 |
| order_status | X | X | 제외 | 주문 이후에 정해지는 정보 |
| order_id, customer_id | O | X | 제외 | 식별자라서 숫자로 넣으면 의미 없는 번호 순서나 특정 고객을 외우게 됨 |
| order_total | - | 타깃 자체 | 제외 | 예측 대상 |

- 모델 입력 6개는 금지 입력값 목록과 겹치지 않으며, validate_feature_columns() 검사를 오류 없이 통과했다. - 모델링 데이터 300행은 주문 상태와 관계없이 모든 주문을 포함한다(order_status는 입력에서만 제외).
- 모델링 데이터(300행)와 주문 파일(300행)의 행 수가 같아, 주문 상세 합계와 고객 정보를 붙이는 병합에서 행이 늘거나 빠지지 않았다.

![Leakage 점검](images/step02_leakage.png)

### 가장 위험하다고 판단한 feature와 이유
item_count, total_quantity, avg_unit_price가 가장 위험하다. line_total이나 quantity처럼 일반 변수처럼 보이지만, 모두 주문 상세에서 계산한 값이다. 특히 total_quantity와 avg_unit_price를 함께 넣으면 둘을 곱해 order_total에 가까운 값을 만들 수 있어, 모델이 예측이 아니라 계산을 하게 된다. 이 값들은 예측 시점에 알 수도 없다.

## 3. 데이터 분할
- train 범위: 2025-09-14 ~ 2026-06-14, 주문 240건 (앞쪽 80%)
- validation 범위: 별도 구간 없음. train 240건 안에서 날짜 순서로 최대 5번 나눈 시간 순서 교차검증 실행.
- final test 범위: 2026-06-14 ~ 2026-09-13, 주문 60건 (뒤쪽 20%)
- 분할 기준: 주문일 순서로 정렬해 앞 80%를 train, 뒤 20%를 final test로 나눴다.

### 나의 해석과 판단
이 모델은 과거 주문으로 이후 주문 금액을 예측하는 상황을 가정하므로 시간 분할을 선택했다. 랜덤 분할시 미래 주문이 train에 섞여, 실제로는 알 수 없는 시기의 패턴으로 학습하게 된다.

- 분할 경계일인 2026-06-14의 주문이 train과 final test 양쪽에 있다. 주문 순서로 나눴기 때문에 중복이 발생했으나 같은 날 주문이 섞이는 정도라 영향은 작다고 판단된다.
- train set에는 9~12월, 1~6월 데이터는 포함하지만 7월과 8월은 없다.
- 노트북의 셀 19은 final test MAE가 가장 낮은 모델을 선택했다. 이 모델은 오차 진단용이지만, final test를 모델 선택에 쓰지 말라는 원칙에 비추어 최종 모델 선택의 근거로는 쓰지 않는다.

## 4. Baseline 비교
| 모델/기준 | MAE | RMSE | R² | 비고 |
| --- | ---: | ---: | ---: | --- |
| Baseline (훈련 평균 예측) | 455,896 | 531,016 | -0.002 | 기준 |
| Linear Regression | 484,513 | 568,269 | -0.148 | MAE가 baseline보다 6.28% 나쁨 |
| Random Forest | 484,884 | 564,053 | -0.131 | MAE가 baseline보다 6.36% 나쁨, train MAE 323,901 |

(final test 60건 기준. 시간 순서 교차검증 MAE 평균: Random Forest 430,353 ± 3,650, Linear Regression 524,294 ± 113,294, 두 모델 모두 R제곱 평균이 음수)

![회귀 평가](images/step04_metrics.png)

### 결과 관찰
- 두 모델 모두 final test MAE와 RMSE가 baseline보다 크고, R제곱이 음수이다. 훈련 평균 하나로 예측하는 것보다 오차가 크다.
- Random Forest는 train MAE(323,901)와 test MAE(484,884)의 차이가 약 16만 원으로, 훈련 데이터에 overfitting 되었을 가능성이 있다.
- 교차검증에서 Linear Regression은 MAE 표준편차가 113,294로 기간에 따라 성능이 크게 흔들렸고, Random Forest는 안정적이지만 R제곱이 음수이다.

### 나의 해석과 판단
모델이 baseline보다 의미 있게 나아지지 않았다. 오히려 두 모델 모두 baseline보다 약 6% 나쁘게 나와 현재 입력값(결제 수단, 주문 시기, 고객 속성)으로는 주문 금액을 평균보다 잘 설명하지 못한다고 판단했다.

### 업무·분석적 의미
baseline의 평균 오차도 약 46만 원으로, 전체 주문의 평균 주문 금액(약 85만 원)의 절반이 넘는다. 개별 주문 금액 예측의 용도로는 쓸 수 없는 수준이며, 이 결과는 "주문 상세 없이는 주문 금액을 예측하기 어렵다"로 해석될 수 있다.

### 한계와 추가 확인 사항
- final test가 60건뿐이라 6% 차이는 몇 건의 큰 오차로 바뀔 수 있다.
- 모델링 데이터에 취소/환불 주문의 금액도 타깃으로 포함되어 있다.
- 교차검증 결과에 baseline이 없어, 교차검증에서도 baseline보다 나쁜지는 확인하지 못했다. -> 실습 과제 3에서 기간별 baseline을 계산한 결과, 교차검증에서도 두 모델 모두 baseline(평균 414,459원)보다 MAE가 컸다.

## 5. 대표 오차 사례
진단 모델: Linear Regression (final test 60건). 절대 오차는 평균 484,513원, 중앙값 495,738원, 최대 1,262,949원이다.

### 사례 1
- 실제값: 1,877,000원
- 예측값: 614,051원
- 오차: +1,262,949원 (실제보다 낮게 예측, 60건 중 가장 큰 오차)
- 특징: 2026-08-21 주문(주문 ID 51). 오차 상위 10건 중 7건이 이처럼 실제값이 150만 원 이상인 주문을 60~100만 원 수준으로 낮게 예측했다.
- 가능한 이유 후보: 주문 금액은 담긴 상품의 수량과 단가로 정해지는데, 입력값(결제 수단, 주문 시기, 고객 속성)에는 이 정보가 없다. 그래서 모델이 큰 주문을 구분하지 못하고 평균에 가까운 값을 예측했을 가능성이 있다.
- 추가 확인: 이 주문의 상품 수와 단가, 같은 고객의 과거 주문 금액이 컸는지 확인한다.

### 사례 2
- 실제값: 10,000원
- 예측값: 994,142원
- 오차: -984,142원 (실제보다 높게 예측)
- 특징: 2026-08-17 주문(주문 ID 168). 실제 금액이 10만 원 미만인 주문 3건(10,000원, 72,000원, 86,000원)이 모두 약 95~100만 원으로 예측되어 오차 상위 10건에 들었다.
- 가능한 이유 후보: 사례 1과 같은 이유(주문 금액은 담긴 상품의 수량과 단가로 정해지는데, 입력값에는 이 정보가 없다)로 작은 주문도 구분하지 못했다.
- 추가 확인: 같은 월의 다른 주문 예측값도 높은지 확인하고, order_month를 숫자가 아닌 범주로 바꾸거나 제외했을 때 예측값이 달라지는지 본다. 이 주문의 상품 수와 단가, 같은 고객의 과거 주문 금액이 작았는지 확인한다.

### 공통 패턴
예측값은 약 61만~101만 원의 좁은 범위에 있는데 실제값은 1만~188만 원으로 넓다. 모델이 금액의 차이를 거의 설명하지 못해, 큰 주문은 낮게, 작은 주문은 높게 예측하는 패턴이다. 예측 시점 이전의 고객 구매 이력(예: 주문일 전까지의 평균 주문 금액)처럼 금액 규모와 관련 있으면서 미래 정보가 섞이지 않는 feature가 필요하다.

## 6. Final Test
- final test 성능: Random Forest (교차검증 MAE가 더 낮아 선택) MAE 484,884원, RMSE 564,053원, R제곱 -0.131. 같은 기간 baseline은 MAE 455,896원, R제곱 -0.002이다.
- validation 대비 변화: 교차검증 MAE 평균 430,353원 -> final test MAE 484,884원으로 약 54,531원(약 12.7%) 커졌다. R제곱은 -0.150 -> -0.131로 둘 다 음수이다.
- 다만 교차검증에서도 baseline(평균 414,459원)이 Random Forest(430,353원)보다 오차가 작아, 교차검증 기준으로는 모델보다 baseline이 낫다.

### 결과 관찰
- 교차검증으로 고른 Random Forest가 final test에서도 baseline보다 MAE가 6.36% 크다.
- final test MAE가 교차검증 평균보다 약 12.7% 크다. 교차검증 MAE의 표준편차(3,650원)보다 차이가 훨씬 크다.
- 교차검증과 final test 모두 R제곱이 음수라, 어느 기간에서도 평균 예측보다 나은 성능을 보이지 못했다.
- Linear Regression은 final test MAE(484,513원)가 교차검증 평균(524,294원)보다 작지만, 교차검증 표준편차가 113,294원으로 커서 이 차이를 개선으로 보기는 어렵다.

### 나의 해석과 판단
차이가 생긴 이유 후보는 다음과 같다.
1. Random Forest는 train MAE(323,901원)가 낮아 훈련 기간에 과하게 맞춰졌을 수 있다.
2. final test가 60건뿐이라 큰 오차 몇 건(5단계 사례)이 평균 오차를 크게 바꿀 수 있다.
3. train에 7,8월 주문이 없어 그 시기의 패턴을 학습할 수 없었다(교차검증에서도 R제곱은 음수였으므로 주된 원인은 아니다).

어느 이유가 더 영향이 큰지는 이 결과만으로 단정할 수 없다. 다만 교차검증과 final test 모두에서 R제곱이 음수였으므로, 기간이 바뀌어서 갑자기 나빠졌다기보다 현재 입력값으로는 주문 금액을 일반적으로 설명하지 못한다고 판단했다.

## 7. 최종 사용 판단
- [ ] 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 사용 보류

### 판단 근거
1. baseline 대비 성능: 교차검증으로 고른 Random Forest와 Linear Regression 모두 final test MAE가 baseline보다 약 6% 크고, R제곱이 교차검증과 final test 모두에서 음수이다. 모델을 쓰는 것이 훈련 평균 하나로 예측하는 것보다 예측 성능이 나쁘다.
2. 오차 크기: baseline도 평균 오차가 약 46만 원으로 평균 주문 금액(약 85만 원)의 절반이 넘는다. 예측값은 약 61만~101만 원의 좁은 범위에 몰려 있어, 1만 원 주문과 188만 원 주문을 구분하지 못한다.
3. 누수 점검과 데이터 범위: 타깃 계산 재료, 주문 이후 정보, 식별자를 모두 제외해 누수는 없다. 그러나 남은 입력값 6개(결제 수단, 주문 시기, 고객 속성)만으로는 금액을 설명하지 못한다. 데이터가 약 1년(300건)뿐이라 train에 7,8월이 없고, final test는 60건이다.

### 업무적 위험
이 모델로 주문 금액을 미리 추정하면 큰 주문은 크게 낮게, 작은 주문은 크게 높게 예측한다. 예를 들어 재고 준비나 매출 예상에 쓰면 실제보다 평균 46만 원 이상 틀린 숫자를 믿게 된다. 또 R제곱이 음수인 모델을 'AI 예측'이라는 이유로 신뢰하면, 단순 평균을 쓸 때보다 판단이 나빠질 수 있다.

### 다음 개선 우선순위
1. 예측 시점 이전의 고객 구매 이력 feature 추가: 주문일 전날까지의 고객별 주문 수와 평균 주문 금액처럼, 금액 규모와 관련 있으면서 미래 정보가 섞이지 않는 값을 만든다. 집계 기준일을 주문일 이전으로 명시한다.
2. 데이터 범위 보완: 모든 월이 train에 들어가도록 더 긴 기간의 데이터를 확보하거나, order_month를 숫자가 아닌 범주로 바꾸거나 제외해 비교한다. 취소·환불 주문을 타깃에 포함할지도 업무 목적에 맞게 정한다.
3. 검증 방식 보완: 교차검증에도 baseline을 넣어 기간별로 비교하고, Random Forest의 과적합을 줄이는 설정(예: min_samples_leaf)을 교차검증으로 고른다.

## 최종 체크
- [x] leakage를 점검했습니다.
- [x] baseline과 비교했습니다.
- [x] validation[[과 final test를 구분했습니다.
- [x] 점수만이 아니라 오차 의미를 해석했습니다.
- [x] 최종 Notebook URL을 제출합니다.